# Find the Order — Baseline solution

A **deterministic** baseline. It uses `prefix.json` — the two chunks known to come
first — and leaves every other chunk in the order it was given.

It writes **`answers.json`** at the repo root in the rank convention: `P[i]` is the
predicted chronological position of `chunk_i.wav`.

Replace the logic below with your own.

In [ ]:
import os, json

TEST_DIR = "dataset/test_public"        # grade-time: hidden eval set is overlaid here
OUTPUT   = "answers.json"

# prefix.json ships with every split. answers.json does NOT exist in the hidden
# grading set -- never read it from TEST_DIR, or your notebook dies at grade time.
with open(os.path.join(TEST_DIR, "prefix.json")) as f:
    prefix = json.load(f)

def n_chunks(ddir):
    return sum(1 for f in os.listdir(ddir)
               if f.startswith("chunk_") and f.endswith(".wav"))

# Only numeric directories are dialogues -- skip strays such as .ipynb_checkpoints,
# which JupyterLab creates as soon as you open something inside the folder.
dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR)
     if d.isdigit() and os.path.isdir(os.path.join(TEST_DIR, d))),
    key=int,
)

answers = {}
for did in dialogue_ids:
    n = n_chunks(os.path.join(TEST_DIR, did))
    first, second = prefix[did]
    order = [first, second] + [i for i in range(n) if i not in (first, second)]
    rank = [0] * n                      # rank[i] = position of chunk_i
    for pos, idx in enumerate(order):
        rank[idx] = pos
    answers[did] = rank

with open(OUTPUT, "w") as f:
    json.dump(answers, f)
print(f"wrote {OUTPUT}: {len(answers)} dialogues")


## Additional Models & Libraries

## wav2-vec2-base-960h

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import torch
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Processor

WAV2VEC_PATH = "models/wav2vec2-base-960h"
SAMPLE_AUDIO = "dataset/train/0/chunk_0.wav"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_PATH)
model = Wav2Vec2Model.from_pretrained(WAV2VEC_PATH).to(device).eval()

audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    h = model(inputs.input_values.to(device)).last_hidden_state
embed = h.mean(dim=1).squeeze().cpu().numpy()

print(f"Wav2vec embedding shape: {embed.shape}")

## Qwen2.5-0.5B

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN25_MODEL_PATH = "models/qwen2.5-0.5b"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(QWEN25_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(QWEN25_MODEL_PATH, dtype=torch.bfloat16).to(device).eval()

prompt = "The weather today is"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=20, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## Whisper-small

In [ ]:
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WHISPER_MODEL_PATH = "models/whisper-small"
SAMPLE_AUDIO = "dataset/train/0/chunk_0.wav"

processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_PATH).to(device).eval()


audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
feats = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
with torch.no_grad():
    ids = model.generate(feats.to(device), language="en", task="transcribe")
result = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
print(f"Whisper ASR trasncription: {result}")